In [1]:
from langchain.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain.prompts import PromptTemplate
from llama_parse import LlamaParse
from langchain_openai import AzureChatOpenAI
from langchain.schema import Document
from langchain.chains import LLMChain, RetrievalQA
from langchain.agents import initialize_agent, Tool
from llama_index.core import Document as llama_Document


C:\Users\raju220738\.conda\envs\playbook\Lib\site-packages\langchain_community\utilities\__init__.py:8: LangChainDeprecationWarning: As of langchain-core 0.3.0, LangChain uses pydantic v2 internally. The langchain_core.pydantic_v1 module was a compatibility shim for pydantic v1, and should no longer be used. Please update the code to import from Pydantic directly.

For example, replace imports like: `from langchain_core.pydantic_v1 import BaseModel`
with: `from pydantic import BaseModel`
or the v1 compatibility namespace if you are working in a code base that has not been fully upgraded to pydantic 2 yet. 	from pydantic.v1 import BaseModel

  from langchain_community.utilities.requests import (


In [2]:
import os
import glob
import nest_asyncio

nest_asyncio.apply()

# API access to llama-cloud
os.environ["LLAMA_CLOUD_API_KEY"] = "llx-OfjtqDQ0kvPFvoBzdf84MgBLNneLKp9OdpBvpxgcb2AgsLSo"

In [3]:
# Using Azure OpenAI API for embeddings/llms
os.environ["AZURE_OPENAI_API_KEY"] = "95d5f4e4a96a46a4ba1900155e8b641c"

llm = AzureChatOpenAI(
    azure_endpoint="https://creditmemo.openai.azure.com/",
    azure_deployment='gpt-4o',
    openai_api_version='2024-02-15-preview')

In [4]:
title_prompt=PromptTemplate.from_template('Generate a concise titles based on the following text :\n{text}')
keywords_prompt=PromptTemplate.from_template('Generate  a list of keywords based on the following text :\n{text}')

In [5]:
title_chain=LLMChain(llm=llm,prompt=title_prompt)
keyword_chain=LLMChain(llm=llm,prompt=keywords_prompt)

C:\Users\raju220738\AppData\Local\Temp\1\ipykernel_14536\2677860139.py:1: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use :meth:`~RunnableSequence, e.g., `prompt | llm`` instead.
  title_chain=LLMChain(llm=llm,prompt=title_prompt)


In [6]:

def generate_metadata(chunk_text):
    title = title_chain.invoke(chunk_text)
    keywords = keyword_chain.invoke(chunk_text)
    # Ensure title and keywords are strings
    title_str = title if isinstance(title, str) else str(title)
    keywords_str = keywords if isinstance(keywords, str) else str(keywords)
    return {'Title': title_str.strip().strip('"'), 'keywords': keywords_str.strip()}


In [9]:
file_lists = [
    ["FS_3Q24_VZ_102224", "Transcript-3Q24-VZ-102224"],
    ["Credit_Ratings", "Fitch"],
    ['Infographic_3Q24_VZ_102224', 'Presentation_3Q24_VZ_102224'],
    ['Verizon-2023-ESG-Report', 'Verizon_Certificate_of_Incorporation', 'Verizon_Institutional Holdings']
]

# Define the source directory
source_docs = r"C:\Users\raju220738\Downloads\source_docs"
for idx, files in enumerate(file_lists):
    all_documents=[]
    for file in files:
        file_with_extension=file+'.pdf'
        pdf_path=os.path.join(source_docs, file_with_extension)
        # initilize llamaParse object
        documents = LlamaParse(result_type="markdown").load_data(pdf_path)
        #extract text and join into single string
        text_data='\n'.join([doc.text for doc in documents if doc.text.strip()])
        #split documents into smaller chunks
        text_splitter = RecursiveCharacterTextSplitter(chunk_size = 1500, chunk_overlap = 300)  
        split_docs = text_splitter.split_documents([Document(page_content=text_data)])
        #append the split documents to all_documents list
        all_documents.extend(split_docs)
    chunked_documents=[]    
    for doc in all_documents:
        metadata=generate_metadata(doc.page_content)
        chunked_documents.append(Document(page_content=doc.page_content,metadata=metadata)) 
        # create embedding and FAISS vector store
    embeddings=HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')  
    # create a new FAISS vector store for this list of files
    print("Raju---------------------------------------------------Raju")
    vectorstore_path=f"c:\\users\\raju220738\\Downloads\\vectorstore_{idx}.faiss"
    vectorstore=FAISS.from_documents(chunked_documents,embeddings)
    # save FAISS index locally with unique path
    vectorstore.save_local(vectorstore_path) 
    

Started parsing the file under job_id 928a3572-caac-44c0-928f-0ffcdb2ec7cd
Started parsing the file under job_id d99075bc-7e58-4ea3-9ba1-7d6b4d74507c
Raju---------------------------------------------------Raju
Started parsing the file under job_id 800a1a83-9182-4dfe-a93d-e1a287a5aab9
Started parsing the file under job_id 1c6311be-e963-4291-81f4-f1b2303bf2ad
Raju---------------------------------------------------Raju
Started parsing the file under job_id 81a35af0-aa0b-4a8f-aab1-f38e70b57936
Started parsing the file under job_id 064aa146-c6e6-4ed5-91c0-d55853188353
Raju---------------------------------------------------Raju
Started parsing the file under job_id b3fb90cb-3f19-462a-a294-7b06fc563fe2
Started parsing the file under job_id 07cbaf06-7d44-4db3-93bc-1d1148365873
Started parsing the file under job_id b90c61a8-7449-47ee-bf73-fd21c84e13d2
Raju---------------------------------------------------Raju


In [34]:
file_lists = [
    ["2023-Annual-Report-on-Form-10k","FS_3Q24_VZ_102224", "Transcript-3Q24-VZ-102224","2024-telecom-industry-outlook"]
]    

# Define the source directory
source_docs = r"C:\Users\raju220738\Downloads\source_docs"
for idx, files in enumerate(file_lists):
    all_documents=[]
    for file in files:
        file_with_extension=file+'.pdf'
        pdf_path=os.path.join(source_docs, file_with_extension)
        # initilize llamaParse object
        documents = LlamaParse(result_type="markdown").load_data(pdf_path)
        #extract text and join into single string
        text_data='\n'.join([doc.text for doc in documents if doc.text.strip()])
        #split documents into smaller chunks
        text_splitter = RecursiveCharacterTextSplitter(chunk_size = 3000, chunk_overlap = 500)  
        split_docs = text_splitter.split_documents([Document(page_content=text_data)])
        #append the split documents to all_documents list
        all_documents.extend(split_docs)
    chunked_documents=[]    
    for doc in all_documents:
        metadata=generate_metadata(doc.page_content)
        chunked_documents.append(Document(page_content=doc.page_content,metadata=metadata)) 
        # create embedding and FAISS vector store
    embeddings=HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')  
    # create a new FAISS vector store for this list of files
    print("Raju---------------------------------------------------Raju")
    vectorstore_path=f"c:\\users\\raju220738\\Downloads\\vectorstore_{idx}.faiss"
    vectorstore=FAISS.from_documents(chunked_documents,embeddings)
    # save FAISS index locally with unique path
    vectorstore.save_local(vectorstore_path) 
    

Started parsing the file under job_id 5268cc93-7632-4b0b-b0b1-b7c27ada626d
Started parsing the file under job_id 19b98a5c-30bf-4432-bcb6-d616c284f6dc
Started parsing the file under job_id ad71fbbc-55b7-4d48-92f8-e516146dfc87
Started parsing the file under job_id fa24240c-b790-419f-9e56-ecf564c8e449
Raju---------------------------------------------------Raju


In [35]:
embeddings=HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')

In [ ]:
reports_and_outlooks_retriever=FAISS.load_local(r"C:\Users\raju220738\Downloads\vectorstore_0.faiss", embeddings).as_retriever(search_kwargs={"k":5})


In [17]:
finanacial_and_credit_info_retriever=FAISS.load_local(r"C:\Users\raju220738\Downloads\vectorstore_1.faiss", embeddings).as_retriever(search_kwargs={"k":5})

In [18]:
presentations_and_infographic_retriever=FAISS.load_local(r"C:\Users\raju220738\Downloads\vectorstore_2.faiss", embeddings).as_retriever(search_kwargs={"k":5})

In [19]:
corporate_documents_retriever=FAISS.load_local(r"C:\Users\raju220738\Downloads\vectorstore_3.faiss", embeddings).as_retriever(search_kwargs={"k":5})

In [20]:
tools=[ Tool(
    name="Reports and Outlooks",
    func=reports_and_outlooks_retriever.get_relevant_documents,
    description="Retrieves relevant reports and outlooks related documents"
),
Tool(
    name="Financial and Credit Info",
    func=finanacial_and_credit_info_retriever.get_relevant_documents,
    description="Retrieves relevant financial and credit info related documents"
),
Tool(
    name="Presentations and Infographic",
    func=presentations_and_infographic_retriever.get_relevant_documents,
    description="Retrieves relevant presentations and infographic related documents"
),
Tool(
    name="Corporate Documents",
    func=corporate_documents_retriever.get_relevant_documents,
    description="Retrieves relevant corporate documents related documents"
)]

In [21]:
agent=initialize_agent(tools,llm,agent="zero-shot-react-description",handle_parsing=True,verbose=True)

C:\Users\raju220738\AppData\Local\Temp\1\ipykernel_14536\638906484.py:1: LangChainDeprecationWarning: LangChain agents will continue to be supported, but it is recommended for new use cases to be built with LangGraph. LangGraph offers a more flexible and full-featured framework for building agents, including support for tool-calling, persistence of state, and human-in-the-loop workflows. For details, refer to the `LangGraph documentation <https://langchain-ai.github.io/langgraph/>`_ as well as guides for `Migrating from AgentExecutor <https://python.langchain.com/docs/how_to/migrate_agent/>`_ and LangGraph's `Pre-built ReAct agent <https://langchain-ai.github.io/langgraph/how-tos/create-react-agent/>`_.
  agent=initialize_agent(tools,llm,agent="zero-shot-react-description",handle_parsing=True,verbose=True)


In [22]:
decomposition_prompt=PromptTemplate.from_template(
    "Break the complex query '{query}' into more than 3 focused sub-questions, without extra context or headings."
)

In [23]:
decomposition_chain=LLMChain(llm=llm,prompt=decomposition_prompt)

In [24]:
response_merging_prompt=PromptTemplate.from_template(
    "Here are partial answers to a complex query:\n\n{sub_answer}\n\n"
    "Based on these partial answers, genrate a single , structured final response."
)

In [25]:
response_merging_chain=LLMChain(llm=llm,prompt=response_merging_prompt)

In [30]:
def process_complex_query(query):
    sub_questions=decomposition_chain.run(query).split("\n")
    sub_questions=[sub_question.strip() for sub_question in sub_questions if sub_question.strip()]
    print("\n Generated sub-questions")
    for i, sub_question in enumerate(sub_questions,1):
        print(f"{i}. {sub_question}")
    sub_answers={sub_q: agent.run(sub_q) for sub_q in sub_questions}
    sub_answers_str = [f"{sub_question}: {sub_answers}" for sub_question in sub_questions]
    final_response = response_merging_chain.run("\n".join(sub_answers_str))
    return final_response

In [31]:
query=["How was the financial performance of Verizon in 2023 compared to performance in 2022?"]

In [32]:
response=process_complex_query(query)


 Generated sub-questions
1. 1. What were Verizon's total revenues in 2023 compared to 2022?
2. 2. How did Verizon's net profit in 2023 compare to its net profit in 2022?
3. 3. What were Verizon's operating expenses in 2023 versus 2022?
4. 4. How did Verizon's earnings per share (EPS) in 2023 compare to 2022?
5. 5. What changes occurred in Verizon's market share between 2022 and 2023?
6. 6. How did Verizon's stock price fluctuate throughout 2023 compared to 2022?


> Entering new AgentExecutor chain...
To answer this question, I will need to find financial documents or reports that contain Verizon's revenue figures for the years 2022 and 2023. I will use the "Financial and Credit Info" tool to retrieve this information.

Action: Financial and Credit Info
Action Input: "Verizon total revenues 2022 2023"
Observation: [Document(metadata={'Title': '{\'text\': \'"Fitch Affirms Verizon\\\'s \\\'A-\\\' Rating; Highlights Market Leadership and Spectrum Acquisition"\'}', 'keywords': '{\'text\':

Thought:The documents retrieved do not contain specific information about Verizon's total revenues for 2022 and 2023. To find this information, I will try using the "Reports and Outlooks" tool to see if there are any financial reports or outlooks that include Verizon's revenue figures for these years.

Action: Reports and Outlooks
Action Input: "Verizon total revenues 2022 2023"
Observation: [Document(metadata={'Title': '{\'text\': \'"Verizon Reports Record Q3 Growth in Wireless Revenue and EBITDA"\'}', 'keywords': "{'text': '1. Superior network  \\n2. Network infrastructure  \\n3. Resilient  \\n4. Customers  \\n5. First responders  \\n6. Third quarter performance  \\n7. Improvements  \\n8. Wireless service revenue  \\n9. Adjusted EBITDA  \\n10. Free cash flow  \\n11. Sequential growth  \\n12. Year-over-year growth  \\n13. Telecom industry  \\n14. Financial guidance  \\n15. Operational performance  \\n16. Mobility  \\n17. Broadband  \\n18. Private networks  \\n19. Strategic M&A  \\n20.

Thought:The observation from the "Reports and Outlooks" tool includes information about Verizon's operating revenues for part of 2023 and the third quarter of 2024, but it does not provide a clear comparison of total annual revenues for 2022 and 2023. To obtain this information, I will use the "Corporate Documents" tool to see if there are any corporate documents or annual reports that detail Verizon's total revenues for those years.

Action: Corporate Documents
Action Input: "Verizon total revenues 2022 2023"
Observation: [Document(metadata={'Title': '{\'text\': \'"Verizon 2023 ESG Report Summary"\'}', 'keywords': "{'text': '1. Verizon  \\n2. 2023  \\n3. ESG  \\n4. Report  \\n5. Year-end  \\n6. Endnotes'}"}, page_content='*As of year-end 2023 unless otherwise noted.\n\nVerizon 2023 ESG Report\n# Verizon 2023 ESG Report\n\n# Endnotes'), Document(metadata={'Title': '{\'text\': \'"Verizon Innovative Learning: Bridging the Digital Divide with Technology-Integrated Education"\'}', 'keyword

Thought:The documents from the "Corporate Documents" tool did not provide the specific total revenue figures for Verizon for 2022 and 2023. Given the lack of precise data in the retrieved documents, it appears I am unable to provide the exact comparison of Verizon's total revenues for these years based on the available resources. 

Final Answer: The available documents do not provide specific total revenue figures for Verizon for 2022 and 2023. Further detailed financial reports or annual reports from Verizon would be needed to obtain this information.

> Finished chain.


> Entering new AgentExecutor chain...
To answer the question about Verizon's net profit comparison between 2023 and 2022, I will need to find relevant financial documents or reports that contain this information. 

Action: Financial and Credit Info
Action Input: "Verizon net profit 2023 vs 2022"
Observation: [Document(metadata={'Title': '{\'text\': \'"Fitch Affirms Verizon\\\'s Stable \\\'A-\\\' Rating Amid Competiti

Thought:The documents retrieved do not provide specific details on Verizon's net profit for 2023 compared to 2022. I will need to search through reports and outlooks to find the necessary financial comparison information.

Action: Reports and Outlooks
Action Input: "Verizon net profit 2023 vs 2022"
Observation: [Document(metadata={'Title': '{\'text\': \'"Verizon Reports Record Q3 Growth in Wireless Revenue and EBITDA"\'}', 'keywords': "{'text': '1. Superior network  \\n2. Network infrastructure  \\n3. Resilient  \\n4. Customers  \\n5. First responders  \\n6. Third quarter performance  \\n7. Improvements  \\n8. Wireless service revenue  \\n9. Adjusted EBITDA  \\n10. Free cash flow  \\n11. Sequential growth  \\n12. Year-over-year growth  \\n13. Telecom industry  \\n14. Financial guidance  \\n15. Operational performance  \\n16. Mobility  \\n17. Broadband  \\n18. Private networks  \\n19. Strategic M&A  \\n20. Cost efficiencies  \\n21. Verizon'}"}, page_content='Our superior network made a 

...

Thought:The documents retrieved still do not provide specific information on Verizon's earnings per share (EPS) for 2023 or 2022. Therefore, I am unable to provide a comparison of Verizon's EPS between those two years based on the current documents. Further detailed financial reports or corporate documents would be needed to obtain this specific information.

Final Answer: I could not find specific information on Verizon's earnings per share (EPS) for 2023 or 2022 in the documents accessed. Additional financial reports would be required to provide this comparison.

> Finished chain.


> Entering new AgentExecutor chain...
To determine the changes in Verizon's market share between 2022 and 2023, I should look for relevant reports and outlooks that might contain market analysis or financial data related to Verizon's market position during this period. 

Action: Reports and Outlooks
Action Input: "Verizon market share 2022 2023"
Observation: [Document(metadata={'Title': '{\'text\': \'"Ver

Thought:The documents provide insights into Verizon's performance and strategic initiatives, but they do not directly mention changes in market share between 2022 and 2023. They focus more on operational performance, financial growth, and strategic plans. However, they do highlight positive growth trends and achievements such as increased wireless service revenue, strong EBITDA, and net additions in the consumer segment, which could indicate an improvement or stabilization in market share.

To better understand the specific changes in Verizon's market share between 2022 and 2023, I would need to find more targeted data or reports explicitly addressing market share statistics or comparisons over these years. Let me search again specifically for Verizon's market share data.

Action: Reports and Outlooks
Action Input: "Verizon market share analysis 2022 2023"
Observation: [Document(metadata={'Title': '{\'text\': \'"Verizon Reports Record Q3 Growth in Wireless Revenue and EBITDA"\'}', 'key

Thought:The documents reviewed do not provide specific statistical data on Verizon's market share changes between 2022 and 2023. They emphasize Verizon's strong operational performance, financial growth, and strategic initiatives in 2023, such as increased wireless service revenue, strategic acquisitions, and partnerships. These elements may suggest a positive trajectory for Verizon's market position, but the exact market share figures are not detailed.

Final Answer: The documents do not provide specific data on Verizon's market share changes between 2022 and 2023. They highlight Verizon's operational and financial growth in 2023, which could imply an improvement or stabilization in market share, but do not offer precise market share statistics.

> Finished chain.


> Entering new AgentExecutor chain...
To answer this question, I need to find detailed information on Verizon's stock price performance throughout 2023 and compare it with 2022. This information is likely found in financia

Thought:The retrieved documents focus more on Verizon's financial outlook, debt, and market position rather than specific stock price fluctuations. To find information specifically about stock price changes, I should look for reports or presentations that analyze stock performance.

Action: Presentations and Infographic
Action Input: "Verizon stock price analysis 2023 2022"
Observation: [Document(metadata={'Title': '{\'text\': \'"Verizon 3Q24 Financial Highlights: Mobility Gains, Broadband Growth, Strategic Execution"\'}', 'keywords': "{'text': '1. SEC rules  \\n2. Non-GAAP financial measures  \\n3. GAAP measures  \\n4. Reconciliation  \\n5. Presentation  \\n6. Verizon  \\n7. Investors  \\n8. Continued momentum  \\n9. 3Q24  \\n10. Mobility improvement  \\n11. Operational performance  \\n12. Financial performance  \\n13. Postpaid phone net adds  \\n14. Year-over-Year (YoY)  \\n15. Prepaid net adds  \\n16. SafeLink  \\n17. Broadband growth  \\n18. FWA subscribers  \\n19. Strategic moves 

Thought:The retrieved documents from the Presentations and Infographic tool focus on Verizon's financial performance, subscriber growth, and strategic initiatives rather than specific stock price fluctuations. The documents do not provide detailed information about how Verizon's stock price specifically fluctuated throughout 2023 compared to 2022.

To accurately answer the question about Verizon's stock price fluctuations over the specified period, I would need access to historical stock price data or a financial analysis report specifically focused on stock performance. Unfortunately, the available documents do not contain this specific information.

Final Answer: The retrieved documents do not provide specific information about Verizon's stock price fluctuations throughout 2023 compared to 2022. To obtain this data, one would need access to historical stock price records or specific financial analysis reports on stock performance.

> Finished chain.


In [33]:
print(response)

Based on the available information, a detailed comparison of Verizon's financial and market performance between 2022 and 2023 is not possible with the current documents. Here is a summary of the key points:

1. **Total Revenues**: Specific figures for Verizon's total revenues in 2022 and 2023 are not provided in the documents. To obtain this information, one would need to refer to Verizon's official financial reports or annual reports.

2. **Net Profit**: The documents do not offer a comparison of Verizon's net profit between 2023 and 2022. Access to Verizon's financial statements or annual reports will be necessary for this data.

3. **Operating Expenses**: Information on Verizon's operating expenses for 2023 versus 2022 is not available in the retrieved documents. The Annual Report or 10-K filings from Verizon would provide precise figures.

4. **Earnings Per Share (EPS)**: The documents lack specific information on Verizon's EPS for both 2023 and 2022. Additional financial reports a